# RDF / OWL → Aura ontology JSON

**Set `INPUT_PATH` and `REDUCE_TO_ROOT_CONCEPTS` below, then Run All.** Use a relative path from this
repository or an absolute local path. Select the `.venv` Python kernel after
installing `requirements.txt`. Output names, serialization, model name and
schema extraction are automatic.

Supported: RDF/XML, Turtle, N-Triples, N3, JSON-LD, TriG, N-Quads and TriX,
including gzip files. `.owl` is treated as a filename extension, not a syntax.
Named graphs are combined for schema extraction. JSON-LD contexts must be embedded;
OWL Functional Syntax and OWL/XML are not RDF serializations.

All inputs use the same declared-class/property and observed-record rules.
No input-specific grouping or relationship whitelist is applied. Declared OWL
classes remain node types; a large ontology can therefore produce a large model.
Unsupported expressions are documented in the report. No instance values are exported.

A live progress indicator shows processing stages and elapsed time, including
during long parsing and extraction steps. The bar measures stages, not a
predicted percentage of bytes or triples processed.

Root reduction is vocabulary-independent. Explicit `has ontology root term` declarations take priority. Otherwise, roots
are active named classes with descendants and no named superclass, ignoring
universal OWL/RDFS top classes.
Their labels become CamelCase; isolated classes and rootless cycles go into
`ExternalConcepts`. Multiple-root classes belong to every reachable root.
Reduction summarizes relationship patterns and metadata properties, and does
not promote member constraints to category-wide constraints.

In [ ]:
INPUT_PATH = 'examples/go-plus.owl'
REDUCE_TO_ROOT_CONCEPTS = True  # False skips scanning and preserves the full schema

In [ ]:
from pathlib import Path
import sys

# Locate the project when the kernel starts in it or one of its subdirectories.
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "rdf_schema.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open this notebook from the repository directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Refresh only project modules, so Run All also works in an existing kernel.
import importlib
for module_name in ("extraction_progress", "rdf_loader", "ontology_converter", "rdf_schema", "root_concepts"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)
from rdf_schema import extract_file
from extraction_progress import NotebookProgress

input_path = Path(INPUT_PATH).expanduser()
if not input_path.is_absolute():
    input_path = project_root / input_path

from rdf_loader import load_rdf_with_metadata
from root_concepts import scan_root_concepts, format_audit_count

## Optional root-concept scan

In [ ]:
# Optional root discovery; disabled mode does not parse or scan here.
prepared_input = None
root_scan = None
if REDUCE_TO_ROOT_CONCEPTS:
    with NotebookProgress(total=3) as progress:
        prepared_input = load_rdf_with_metadata(input_path, progress=progress)
        progress(3, "Discovering named root classes")
        root_scan = scan_root_concepts(prepared_input[0])
    root_labels = [root["label"] for root in root_scan["roots"]]
    print(f"Total classes scanned: {format_audit_count(root_scan['classes_scanned'])}")
    print(f"Active classes considered: {format_audit_count(root_scan['active_classes_scanned'])}; excluded: {root_scan['excluded_classes']:,}")
    print(f"RDF triples scanned: {format_audit_count(root_scan['triples_scanned'])}")
    print(f"Root discovery source: {root_scan['root_source']}")
    print(f"Discovered {len(root_labels)} root concepts:")
    for label in root_labels:
        print(f"  • {label}")
    print(f"Orphan classes: {len(root_scan['orphan_classes']):,} → ExternalConcepts")
    print(f"Classes reaching multiple roots: {len(root_scan['multiple_root_classes']):,}")
    if not root_labels:
        print("No hierarchy roots found; classes will be grouped as ExternalConcepts.")
else:
    print("Root scan skipped. Full schema extraction is unchanged.")

## Extract and save

In [ ]:
with NotebookProgress() as progress:
    model, report, upload_path, report_path = extract_file(
        input_path, progress=progress,
        reduce_to_root_concepts=REDUCE_TO_ROOT_CONCEPTS,
        prepared_input=prepared_input, root_scan=root_scan)
print(f"Input format: {report['input']['format']}")
print(f"Extraction mode: {report['profile']}")
print(f"Validated {len(model['nodes']):,} node types and {len(model['relationships']):,} relationship definitions")
print(f"Upload JSON: {upload_path} ({upload_path.stat().st_size:,} bytes)")
print(f"Review report: {report_path}")

Upload the generated **`.ontology.json`** file in Aura. The `.report.json` is
for review only. Outputs go in this repository's `output/` directory. A short
input-path hash prevents files with identical names in different directories
from overwriting one another; rerunning the same path replaces its outputs.

Inspect the report for inferred mappings and unsupported expressions. Schema
validation does not verify Aura acceptance. This is a schema projection, not a
lossless OWL translation or reasoner. Imports are not fetched. OWL keys and
biological cardinalities do not imply database property constraints. Supported
SHACL literal-property `minCount` rules can create existence constraints.

The GO-plus input is approximately 227 MB and needs considerably more RAM while
parsing. Actual records and values require a separate Neo4j data import.

Reduced outputs include `.reduced.` in the filename, for example
`go-plus-<path-hash>.reduced.ontology.json`; the companion report uses
`go-plus-<path-hash>.reduced.ontology.report.json`.